# Matrix factorization with focal loss: positive vs negative, with negative sampling

Logistic matrix factorization of the **user × recipe** matrix, trained with a class-weighted **focal loss** and fitted by
**alternating Newton steps** (ALS-style: users, then recipes), as a **binary** task: does the user like the recipe?

**Data** (`input_stage2/recipe_user_ratings.csv`)

- Every rating 0–5 is kept (0 is a review without stars).
- **Labels:** a rating of 4–5★ is **positive** (1); a rating of 0–3★ is **negative** (0).
- If a user rated the same recipe more than once, only the latest rating is kept.
- Only users with at least 6 ratings are kept. Every row of the other users is discarded.
- **Split**, per user by date: the latest rating is **test**, the one before it is **valid**, and the rest (at least 4) are **train**.
  Valid picks the regularization strengths and the iteration to keep. Test is used only for the final report.
- **Negative sampling:** every positive rating is paired with 2 recipes the user never rated (1 positive : 2 negatives). They are labelled
  0 (negative) and go to the positive's split. See the negative sampling section.

**Model and loss**

$$\eta_{ui} = \mu + b_u + c_i + p_u^\top q_i, \qquad \hat p_{ui} = \sigma(\eta_{ui})$$

$\eta_{ui}$ is the logit and $\hat p_{ui}$ the probability of a positive. $\mu$ is the logit of the weighted train positive rate, $b_u$ and $c_i$
are the user and recipe biases, and $p_u, q_i \in \mathbb{R}^k$ are the latent factors. Training minimizes a class-weighted **focal loss**
plus an **L2 penalty**:

$$L = \sum_{(u,i) \in \text{train}} w_{y_{ui}}\, \underbrace{\big(1 - p_t\big)^{\gamma}}_{\text{modulating factor}} \big(-\log p_t\big) + \lambda \Big(\sum_u \lVert p_u \rVert^2 + \sum_i \lVert q_i \rVert^2\Big) + \lambda_b \Big(\sum_u b_u^2 + \sum_i c_i^2\Big),
\qquad p_t = \begin{cases} \hat p_{ui} & y_{ui} = 1 \\ 1 - \hat p_{ui} & y_{ui} = 0 \end{cases}$$

$-\log p_t$ is the binary cross-entropy (BCE). The **modulating factor** $(1 - p_t)^{\gamma}$ ($\gamma$ = `FOCAL_GAMMA`) scales it down for rows the
model already gets right ($p_t$ near 1), so training concentrates on the hard ones. $\gamma = 0$ is plain BCE.

**Class weights** ($\alpha$ in the focal loss). With 2 sampled negatives per positive, most train rows are negative.
A row of class $c$ gets the weight $w_c \propto \big(n / (2\, n_c)\big)^{\beta}$, scaled so that the mean train weight is 1. Here $n_c$ is the number of
train rows of class $c$, $n$ is the number of all train rows, and $\beta$ = `WEIGHT_BALANCE`. $\beta = 0$ is unweighted. $\beta = 1$ is scikit-learn's
`"balanced"`: both classes carry the same total weight. Values in between balance partly.

**Fitting: alternating Newton steps.** Squared error has a closed-form ALS step; the focal loss does not. With every recipe's $(q_i, c_i)$
fixed, $L$ still splits into one small problem per user, and a **Newton step** on it is a weighted ridge regression (iteratively
reweighted least squares). With $g$ and $h$ the first and second derivatives of the focal loss with respect to $\eta$ at the current model,
the quadratic approximation of $L$ is minimized by

$$\begin{bmatrix} p_u \\ b_u \end{bmatrix} = \big(Z_u^\top \Omega_u Z_u + \Lambda\big)^{-1} Z_u^\top \Omega_u\, t_u, \qquad t = \eta - \mu - c_i - g / h, \qquad \Omega = \operatorname{diag}\!\big(w\, h / 2\big)$$

with $Z_u = \begin{bmatrix} q_i^\top & 1 \end{bmatrix}_{i \in I_u}$ and $\Lambda = \operatorname{diag}(\lambda, \dots, \lambda, \lambda_b)$, the same batched solve as least-squares ALS.
The recipe step is the same with users and recipes swapped. The focal loss is not convex for $\gamma > 0$ ($h$ turns negative on badly
misclassified rows), so $h$ is floored at `MIN_CURVATURE`, and each step is followed by a **backtracking line search** that keeps only a
step size that does not raise $L$. So $L$ never increases.

**Why the penalty matters.** Without it, a recipe with a single train row could fit that label exactly (its logit would run off to
$\pm\infty$). With it, parameters backed by little weight stay close to 0. A small $\lambda$ gives a low train loss but a high valid loss
(overfitting). A very large $\lambda$ shrinks every factor to 0, which leaves the biases-only model.

**Metrics.** A probability of at least `DECISION_THRESHOLD` (0.5) is predicted **positive**. For each class (negative and positive):
**precision**, **recall** and **F1**; also accuracy and macro F1, and from the probability **ROC AUC** and **average precision**.
The probability itself is scored by the **log loss** (BCE) and a **weighted log loss**: the mean of the per-class log losses, so both
classes count the same however rare. The weighted log loss depends on neither `WEIGHT_BALANCE` nor `FOCAL_GAMMA`, so runs with
different class weights or $\gamma$ stay comparable.

**Tuning.** $\lambda_b$ is picked first on a **biases-only** model ($k = 0$), which is also the bias baseline.
Then $\lambda$ is picked for the $k$-factor model with that $\lambda_b$. Both choices, and early stopping, use the valid metric `SELECT_BY`
(weighted log loss by default). A recipe with no train row gets $q_i = 0$ and $c_i = 0$, so its logit falls back to $\mu + b_u$.


In [ ]:
# Run once if these packages are missing, then restart the notebook kernel.
# %pip install numpy pandas scipy scikit-learn matplotlib
import json
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.special import expit, log_expit, logit
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score)

In [ ]:
RANDOM_STATE = 42
NOTEBOOK_DIR = next((p for p in (Path.cwd(), Path.cwd() / "DataMappingRecommendation")
                     if (p / "input_stage2/recipe_user_ratings.csv").is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Run from DataMappingRecommendation or its project root.")
DATA_PATH = NOTEBOOK_DIR / "input_stage2/recipe_user_ratings.csv"
ALS_DIR = NOTEBOOK_DIR / "model_artifacts/mf_als"

POSITIVE_MIN = 4  # Ratings 4-5 are positive (label 1), ratings 0-3 negative (label 0). Every rating 0-5 is kept (0 = a review without stars).
LABELS = [0, 1]  # negative, positive. Scores are clipped to [0, 1].
MIN_USER_RATINGS = 6  # Users with fewer ratings are discarded, with all their rows.
NEGATIVES_PER_POSITIVE = 2  # Each positive rating is paired with this many recipes the user never rated, labelled 0.
DECISION_THRESHOLD = 0.5  # A clipped score at or above this is predicted positive.
WEIGHT_BALANCE = 1.0  # beta: class weight (alpha) (n / (2 n_c)) ** beta. 0 = unweighted, 1 = "balanced" (both classes weigh the same).
FOCAL_GAMMA = 2.0  # gamma: the focal loss modulating factor (1 - p_t) ** gamma. 0 = plain binary cross-entropy.
SELECT_BY = "weighted log loss"  # The valid metric that picks lambda_b, lambda, and the kept iteration: "weighted log loss" or "log loss".

FACTORS = 32  # k: the size of p_u and q_i.
L2_BIAS_GRID = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]  # lambda_b candidates, tried on the biases-only model.
# The focal loss is ~(1 - p_t)^gamma times smaller than BCE (1/4 at p_t = 0.5 with gamma = 2), so the same lambda penalizes
# relatively more than with BCE or squared error: the grids reach lower values.
L2_GRID = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]  # lambda candidates for the factors.
INIT_STD = 0.1  # q_i starts as N(0, INIT_STD^2); the first user step solves the user side from it.
MIN_CURVATURE = 1e-6  # Floor on the focal loss's second derivative h (it turns negative for badly misclassified rows).
LINE_SEARCH_STEPS = (1.0, 0.5, 0.25, 0.125, 0.0625)  # Newton step sizes tried in order; the first that does not raise L is kept.
MAX_ITERATIONS = 40  # One iteration is a user step followed by a recipe step (Newton steps from small factors start slowly).
EARLY_STOPPING_PATIENCE = 3  # Stop after this many iterations without a better valid SELECT_BY,
MIN_IMPROVEMENT = 1e-4  # where better means lower by more than this.
CHUNK_ELEMENTS = 2**22  # At most this many float64 values (32 MB) per array in a batch of normal equations.
LOGIT_CHUNK = 2**19  # Rows per chunk when computing logits (bounds the gathered factor arrays).
print(f"Ratings: {DATA_PATH}\nArtifacts: {ALS_DIR}")

## Data: every rating 0–5, positive (4–5★) or negative (0–3★), users with at least 6 ratings, split per user by date


In [ ]:
source = pd.read_csv(DATA_PATH, usecols=["user_id", "recipe_id", "rating", "date"])
rated = source.assign(date=lambda frame: pd.to_datetime(frame["date"], format="mixed"))  # Every rating 0-5 is kept.
# One rating per (user, recipe): the latest.
ratings = rated.sort_values("date", kind="stable").drop_duplicates(["user_id", "recipe_id"], keep="last")
kept = ratings[ratings["user_id"].map(ratings["user_id"].value_counts()) >= MIN_USER_RATINGS]
print(f"{len(source):,} rows (ratings 0-5): {len(rated) - len(ratings):,} repeated (user, recipe) pairs merged -> "
      f"{len(ratings):,} ratings from {ratings['user_id'].nunique():,} users")
print(f"Users with >= {MIN_USER_RATINGS} ratings: {kept['user_id'].nunique():,} kept, with {len(kept):,} ratings on "
      f"{kept['recipe_id'].nunique():,} recipes; {ratings['user_id'].nunique() - kept['user_id'].nunique():,} users "
      f"and their {len(ratings) - len(kept):,} ratings discarded")

# Per user, by date: the latest rating is test, the one before it valid, the rest train. Same-day ties keep file order.
kept = kept.sort_values(["user_id", "date"], kind="stable").reset_index(drop=True)
from_end = kept.groupby("user_id").cumcount(ascending=False).to_numpy()  # 0 = the user's latest rating
positive = kept["rating"].to_numpy() >= POSITIVE_MIN
kept = kept.assign(split=np.select([from_end == 0, from_end == 1], ["test", "valid"], "train"),
                   label=positive.astype(np.int8),
                   source=np.where(positive, f"rated {POSITIVE_MIN}-5", f"rated 0-{POSITIVE_MIN - 1}"))
print(f"Observed ratings: {kept['label'].mean():.1%} positive ({POSITIVE_MIN}-5★), "
      f"{1 - kept['label'].mean():.1%} negative (0-{POSITIVE_MIN - 1}★)")


## Negative sampling

Every positive rating is paired with 2 recipes (`NEGATIVES_PER_POSITIVE`, so 1 positive : 2 sampled negatives) that the user never
rated. They are labelled 0 (negative) and get the positive's split and date. The recipes are drawn uniformly from the recipes of the kept
ratings, without repeats for a user and never one the user rated in any split, so a sampled pair never collides with a real rating and
never appears in two splits.

The table shows the share of positives and negatives in each split, before sampling (observed ratings only) and after it, with the
negatives broken down into real low ratings (0–3★) and sampled unrated recipes.


In [ ]:
def sample_unrated(rng, n, rated_positions, pool_size):
    """n distinct random positions in the recipe pool, none in `rated_positions`: draw, drop rated recipes and repeats, top up."""
    chosen = np.empty(0, dtype=np.int64)
    while len(chosen) < n:
        draw = rng.integers(0, pool_size, size=2 * (n - len(chosen)) + 16)
        chosen = pd.unique(np.concatenate([chosen, draw[~np.isin(draw, rated_positions)]]))  # keeps the draw order
    return chosen[:n]


rng = np.random.default_rng(RANDOM_STATE)
recipe_pool = np.sort(kept["recipe_id"].unique())
rated_positions = np.searchsorted(recipe_pool, kept["recipe_id"].to_numpy())  # every kept recipe is in the pool
rated_rows = kept.groupby("user_id").indices  # user -> rows of all their ratings (every split)
positives = kept[kept["label"] == 1]
negatives_per_positive = np.full(len(positives), NEGATIVES_PER_POSITIVE)

owners, sampled_recipes = [], []  # for each sampled recipe: the row (in `positives`) of the positive it is paired with
for user, rows in positives.groupby("user_id").indices.items():
    counts = negatives_per_positive[rows]
    picked = sample_unrated(rng, counts.sum(), rated_positions[rated_rows[user]], len(recipe_pool))
    owners.append(np.repeat(rows, counts))
    sampled_recipes.append(recipe_pool[picked])
negatives = positives.iloc[np.concatenate(owners)][["user_id", "date", "split"]].assign(
    recipe_id=np.concatenate(sampled_recipes), rating=np.nan, label=np.int8(0), source="sampled (unrated)")
labelled = pd.concat([kept, negatives[kept.columns]], ignore_index=True)
assert not labelled.duplicated(["user_id", "recipe_id"]).any()  # no sampled pair repeats a rating or another sample
print(f"{len(negatives):,} sampled negatives for {len(positives):,} positives "
      f"({len(negatives) / len(positives):.2f} per positive) -> {len(labelled):,} labelled rows")

splits = {name: labelled[labelled["split"] == name].reset_index(drop=True) for name in ("train", "valid", "test")}
train_recipe_ids = set(splits["train"]["recipe_id"])
observed_negative, sampled_negative = f"rated 0-{POSITIVE_MIN - 1}", "sampled (unrated)"
class_balance = pd.DataFrame({
    name: {"rows": len(frame), "users": frame["user_id"].nunique(), "recipes": frame["recipe_id"].nunique(),
           "positive": frame["label"].mean(), "negative": 1 - frame["label"].mean(),
           f"negative: {observed_negative}": frame["source"].eq(observed_negative).mean(),
           f"negative: {sampled_negative}": frame["source"].eq(sampled_negative).mean(),
           "negatives per positive": (frame["label"] == 0).sum() / max(frame["label"].sum(), 1),
           "recipe not in train": (~frame["recipe_id"].isin(train_recipe_ids)).mean()}
    for name, frame in {"observed ratings only": kept, **splits, "all (after sampling)": labelled}.items()}).T
share_columns = ["positive", "negative", f"negative: {observed_negative}", f"negative: {sampled_negative}",
                 "recipe not in train"]
display(class_balance.style.format({"rows": "{:,.0f}", "users": "{:,.0f}", "recipes": "{:,.0f}",
                                    "negatives per positive": "{:.2f}"} | {c: "{:.1%}" for c in share_columns}))
print(f"After sampling: {labelled['label'].mean():.1%} positive, {1 - labelled['label'].mean():.1%} negative "
      f"(before: {kept['label'].mean():.1%} positive, {1 - kept['label'].mean():.1%} negative)")


## Label matrix

Users and recipes are indexed from **train** only. `R` is the train label matrix (users × recipes, CSR): 1 for a positive, 0 for a
negative (a 0–3★ rating or a sampled unrated recipe). The zeros are stored explicitly, since they are training rows, and its transpose
(recipes × users) drives the recipe step. A valid or test recipe that has no train row gets index −1.


In [ ]:
user_ids = np.sort(splits["train"]["user_id"].unique())
recipe_ids = np.sort(splits["train"]["recipe_id"].unique())
user_index, recipe_index = pd.Index(user_ids), pd.Index(recipe_ids)


def encode(frame):
    # (user index, recipe index, 0/1 label) arrays; a user or recipe that is not in train gets index -1.
    return (user_index.get_indexer(frame["user_id"]), recipe_index.get_indexer(frame["recipe_id"]),
            frame["label"].to_numpy(np.float64))


encoded = {name: encode(frame) for name, frame in splits.items()}
train_users, train_recipes, train_values = encoded["train"]
R = sp.csr_matrix((train_values, (train_users, train_recipes)), shape=(len(user_ids), len(recipe_ids)))
assert R.nnz == len(splits["train"])  # every train row is stored, the 0 labels included
per_user, per_recipe = np.diff(R.indptr), np.diff(R.tocsc().indptr)
print(f"R: {R.shape[0]:,} users x {R.shape[1]:,} recipes, {R.nnz:,} train rows, {R.sum():,.0f} positive "
      f"(density {R.nnz / np.prod(R.shape):.3%})")
print(f"Train rows per user: min {per_user.min()}, median {np.median(per_user):.0f}, max {per_user.max():,}; "
      f"per recipe: median {np.median(per_recipe):.0f}, max {per_recipe.max():,}, "
      f"{np.mean(per_recipe == 1):.0%} of recipes have one")

# Class weights w_c from the train counts n_c; class_weight[c] is the weight of a row labelled c.
train_counts = np.bincount(train_values.astype(int), minlength=len(LABELS))
class_weight = (len(train_values) / (len(LABELS) * train_counts)) ** WEIGHT_BALANCE
class_weight /= class_weight[train_values.astype(int)].mean()  # The mean train weight is 1.
print(f"Class weights (WEIGHT_BALANCE = {WEIGHT_BALANCE:g}): negative {class_weight[0]:.3f}, "
      f"positive {class_weight[1]:.3f}; weighted train mean {np.average(train_values, weights=class_weight[train_values.astype(int)]):.3f}")


## Focal-loss matrix factorization: alternating Newton steps with L2 regularization and class weights

- `focal_terms` gives the focal loss of every row and its first and second derivatives with respect to the logit.
- `ridge_rows` solves a weighted ridge regression for every row of a sparse target matrix, meaning every user or every recipe.
  Rows with the same number of entries $c$ are stacked into a `[rows, c, k + 1]` array, so one batched matmul builds their
  $Z^\top \Omega Z$ and one `np.linalg.solve` call solves them.
- `newton_half_step` is one side's update: the Newton step as weighted ridge regressions, then the backtracking line search.
- `fit_mf` alternates the user and recipe steps. After each iteration it records the loss $L$ (and warns if it rises), and the
  log loss and weighted log loss on train and valid. It returns the iteration with the best valid `SELECT_BY`, and it stops after
  `EARLY_STOPPING_PATIENCE` iterations without an improvement.


In [ ]:
def ridge_rows(targets, weights, design, reg):
    """One ALS half-step: a weighted ridge regression for every row of the CSR matrix `targets`.

    With t_r the row's stored values, w_r their weights (`weights` is aligned with targets.data), and
    Z_r = design[the row's columns], row r gets
        x_r = argmin_x sum_j w_rj (t_rj - Z_rj x)^2 + sum_d reg_d * x_d^2 = (Z_r^T W_r Z_r + diag(reg))^-1 Z_r^T W_r t_r.
    """
    d = design.shape[1]
    solution = np.zeros((targets.shape[0], d))  # A row without ratings keeps x = 0.
    counts = np.diff(targets.indptr)
    for c in np.unique(counts[counts > 0]):
        same = np.flatnonzero(counts == c)
        step = max(1, CHUNK_ELEMENTS // ((c + d) * d))  # Rows per batch, to bound memory.
        for start in range(0, len(same), step):
            rows = same[start:start + step]
            at = targets.indptr[rows][:, None] + np.arange(c)  # [rows, c]: where each row's ratings are stored
            Z = design[targets.indices[at]]  # [rows, c, d]
            WZ = Z * weights[at][..., None]  # W_r Z_r
            gram = np.swapaxes(WZ, 1, 2) @ Z + np.diag(reg)  # Z^T W Z + diag(reg)
            rhs = np.einsum("rcd,rc->rd", WZ, targets.data[at])  # Z^T W t
            solution[rows] = np.linalg.solve(gram, rhs[..., None])[..., 0]
    return solution


def logits(model, users, recipes):
    # eta = mu + b_u + c_i + p_u . q_i, in chunks of LOGIT_CHUNK rows. A user or recipe outside train (index -1) adds nothing.
    eta = np.full(len(users), model["mu"])
    known_user, known_recipe = users >= 0, recipes >= 0
    eta[known_user] += model["user_bias"][users[known_user]]
    eta[known_recipe] += model["item_bias"][recipes[known_recipe]]
    both = np.flatnonzero(known_user & known_recipe)
    for start in range(0, len(both), LOGIT_CHUNK):
        rows = both[start:start + LOGIT_CHUNK]
        eta[rows] += np.einsum("nk,nk->n", model["user_factors"][users[rows]], model["item_factors"][recipes[rows]])
    return eta


def predict(model, users, recipes):
    # The probability of a positive: sigmoid of the logit.
    return expit(logits(model, users, recipes))


def focal_terms(eta, y):
    """Focal loss of every row, and its first and second derivatives with respect to the logit eta.

    With s = +1 for a positive and -1 for a negative, u = s * eta and p_t = sigmoid(u) (the probability of the true class):
        FL = -(1 - p_t)^gamma log p_t,     dFL/du = (1 - p_t)^gamma (gamma p_t log p_t - (1 - p_t)),
    and d/deta = s d/du, d2/deta2 = d2/du2. gamma = 0 gives binary cross-entropy: g = p - y, h = p (1 - p).
    """
    s = 2 * y - 1
    p_t, one_minus_p_t, log_p_t = expit(s * eta), expit(-s * eta), log_expit(s * eta)
    modulating = one_minus_p_t ** FOCAL_GAMMA
    inner = FOCAL_GAMMA * p_t * log_p_t - one_minus_p_t
    loss = -modulating * log_p_t
    grad = s * modulating * inner
    hess = modulating * (p_t * one_minus_p_t * (FOCAL_GAMMA * (log_p_t + 1) + 1) - FOCAL_GAMMA * p_t * inner)
    return loss, grad, hess


def probability_metrics(y, p):
    # "log loss": binary cross-entropy of the probability. "weighted log loss": the mean of the per-class log losses, so both
    # classes weigh the same however rare. "focal loss": the unweighted mean focal loss with FOCAL_GAMMA.
    p = np.clip(p, 1e-7, 1 - 1e-7)
    bce = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    label = y.astype(int)
    per_class = [bce[label == c].mean() for c in LABELS if np.any(label == c)]
    return {"log loss": float(bce.mean()), "weighted log loss": float(np.mean(per_class)),
            "focal loss": float(focal_terms(logit(p), y)[0].mean())}


def classification_metrics(y, score):
    # A probability >= DECISION_THRESHOLD is predicted positive: per-class precision / recall / F1, accuracy and
    # macro F1; ROC AUC and average precision (positive class) come from the probability itself.
    label, predicted = y.astype(int), (score >= DECISION_THRESHOLD).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(label, predicted, labels=LABELS, zero_division=0)
    both_classes = len(np.unique(label)) == 2
    return {"accuracy": float(np.mean(predicted == label)), "macro f1": float(f1.mean()),
            **{f"{name} {metric}": float(values[c]) for c, name in enumerate(["negative", "positive"])
               for metric, values in (("precision", precision), ("recall", recall), ("f1", f1))},
            "roc auc": float(roc_auc_score(label, score)) if both_classes else np.nan,
            "average precision": float(average_precision_score(label, score)) if both_classes else np.nan}


SIDES = {"user": ("user_factors", "user_bias", "item_factors", "item_bias"),
         "recipe": ("item_factors", "item_bias", "user_factors", "user_bias")}


def newton_half_step(model, side, M, owners, weight, l2, l2_bias, loss_now, objective):
    """One side's update. `M` is R (side "user") or its transpose (side "recipe"); `owners` is the row of every stored entry.

    Newton step on L for [factors, bias] of every row, as weighted ridge regressions on the working response
    t = eta - mu - (other side's bias) - g / h with weights w h / 2, then a backtracking line search: the first step size
    in LINE_SEARCH_STEPS that does not raise L is kept (none: the side is left as it was).
    """
    own_factors, own_bias, other_factors, other_bias = SIDES[side]
    k = model[own_factors].shape[1]
    users, recipes = (owners, M.indices) if side == "user" else (M.indices, owners)
    eta = logits(model, users, recipes)
    _, grad, hess = focal_terms(eta, M.data)
    hess = np.maximum(hess, MIN_CURVATURE)
    target = eta - model["mu"] - model[other_bias][M.indices] - grad / hess
    design = np.c_[model[other_factors], np.ones(M.shape[1])]
    current = np.c_[model[own_factors], model[own_bias]]
    newton = ridge_rows(sp.csr_matrix((target, M.indices, M.indptr), shape=M.shape), weight * hess / 2, design,
                        np.r_[np.full(k, l2), l2_bias])
    for step in LINE_SEARCH_STEPS:
        candidate = current + step * (newton - current)
        trial = {**model, own_factors: candidate[:, :k], own_bias: candidate[:, k]}
        loss = objective(trial)
        if loss <= loss_now:
            return trial, loss
    return model, loss_now


def fit_mf(R, valid, factors, l2, l2_bias, name):
    n_users, n_recipes = R.shape
    Rt = R.T.tocsr()  # Recipes x users, for the recipe step.
    users = np.repeat(np.arange(n_users), np.diff(R.indptr))  # The user of each entry of R.
    recipes_t = np.repeat(np.arange(n_recipes), np.diff(Rt.indptr))  # The recipe of each entry of Rt.
    weight, weight_t = class_weight[R.data.astype(int)], class_weight[Rt.data.astype(int)]  # w of each stored label
    model = {"mu": float(logit(np.average(R.data, weights=weight))),
             "user_factors": np.zeros((n_users, factors)), "user_bias": np.zeros(n_users),
             "item_factors": np.random.default_rng(RANDOM_STATE).normal(0, INIT_STD, (n_recipes, factors)),
             "item_bias": np.zeros(n_recipes)}

    def objective(candidate):
        # L: class-weighted focal loss on train plus the L2 penalties.
        focal = focal_terms(logits(candidate, users, R.indices), R.data)[0]
        factor_penalty = (candidate["user_factors"] ** 2).sum() + (candidate["item_factors"] ** 2).sum()
        bias_penalty = (candidate["user_bias"] ** 2).sum() + (candidate["item_bias"] ** 2).sum()
        return float(weight @ focal + l2 * factor_penalty + l2_bias * bias_penalty)

    history, best, best_model, stale, loss = [], np.inf, None, 0, objective(model)
    for iteration in range(1, MAX_ITERATIONS + 1):
        started, previous = time.time(), loss
        model, loss = newton_half_step(model, "user", R, users, weight, l2, l2_bias, loss, objective)
        model, loss = newton_half_step(model, "recipe", Rt, recipes_t, weight_t, l2, l2_bias, loss, objective)
        if loss > previous * (1 + 1e-6):  # The line search only keeps steps that do not raise L.
            warnings.warn(f"{name}: the loss rose at iteration {iteration} ({previous:.6g} -> {loss:.6g}).")
        row = {"iteration": iteration, "loss": loss}
        for split, (split_users, split_recipes, values) in (("train", (users, R.indices, R.data)), ("valid", valid)):
            metrics = probability_metrics(values, predict(model, split_users, split_recipes))
            row.update({f"{split} log loss": metrics["log loss"], f"{split} weighted log loss": metrics["weighted log loss"]})
        history.append({**row, "seconds": time.time() - started})
        if row[f"valid {SELECT_BY}"] < best - MIN_IMPROVEMENT:
            best, best_model, stale = row[f"valid {SELECT_BY}"], {**model, "iterations": iteration}, 0
        else:
            stale += 1
            if stale >= EARLY_STOPPING_PATIENCE:
                break
    history = pd.DataFrame(history)
    best_row = history.iloc[best_model["iterations"] - 1]
    print(f"{name}: valid log loss {best_row['valid log loss']:.4f}, weighted {best_row['valid weighted log loss']:.4f} "
          f"(train {best_row['train log loss']:.4f}, weighted {best_row['train weighted log loss']:.4f}) "
          f"at iteration {best_model['iterations']} of {len(history)}, {history['seconds'].sum():.0f}s")
    return best_model, history


## Choosing $\lambda_b$ and $\lambda$ on valid

1. **Biases only** ($k = 0$), for every $\lambda_b$ in `L2_BIAS_GRID`. The best one is the bias baseline and fixes $\lambda_b$.
2. **Focal MF with $k$ = `FACTORS`**, for every $\lambda$ in `L2_GRID`, with that $\lambda_b$.

The best run is the one with the lowest valid `SELECT_BY`. Each table row is the kept iteration of one run.

In [ ]:
METRIC_COLUMNS = [f"{split} {metric}" for split in ("train", "valid") for metric in ("log loss", "weighted log loss")]


def search_table(runs, name):
    # One row per candidate: the kept iteration and its train and valid metrics.
    return pd.DataFrame([{name: value, "kept iteration": model["iterations"], "iterations run": len(history),
                          **history.iloc[model["iterations"] - 1][METRIC_COLUMNS].to_dict()}
                         for value, (model, history) in runs.items()]).set_index(name)


valid_rows = encoded["valid"]
bias_runs = {l2_bias: fit_mf(R, valid_rows, 0, 0.0, l2_bias, f"biases only, λ_b={l2_bias:g}")
             for l2_bias in L2_BIAS_GRID}
bias_search = search_table(bias_runs, "λ_b")
best_l2_bias = float(bias_search[f"valid {SELECT_BY}"].idxmin())
als_runs = {l2: fit_mf(R, valid_rows, FACTORS, l2, best_l2_bias, f"focal MF k={FACTORS}, λ={l2:g}") for l2 in L2_GRID}
als_search = search_table(als_runs, "λ")
best_l2 = float(als_search[f"valid {SELECT_BY}"].idxmin())
display(bias_search.round(4), als_search.round(4))
for label, value, grid in (("λ_b", best_l2_bias, L2_BIAS_GRID), ("λ", best_l2, L2_GRID)):
    if value in (min(grid), max(grid)):
        print(f"{label} = {value:g} is at the edge of its grid; extend the grid to check that it is the best.")
print(f"Chosen by valid {SELECT_BY}: λ_b = {best_l2_bias:g}, λ = {best_l2:g}")

### Train and valid loss by $\lambda$

The loss is `SELECT_BY`, the metric that picks $\lambda$. Where the valid loss rises while the train loss keeps falling
(small $\lambda$), the factors overfit. The dashed line is the biases-only model's valid loss, which the factor model
approaches as $\lambda$ grows.

In [ ]:
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
VIZ_STYLE = {
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.titlesize": 12,
    "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
}
metric_label = SELECT_BY

lambdas = als_search.index.to_numpy()
with plt.rc_context(VIZ_STYLE):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.hlines(bias_search.loc[best_l2_bias, f"valid {SELECT_BY}"], lambdas[0], lambdas[-1], colors=MUTED,
              linewidths=1.2, linestyles="--", label=f"biases only, valid (λ_b = {best_l2_bias:g})")
    for split, color in (("valid", BLUE), ("train", ORANGE)):
        column = f"{split} {SELECT_BY}"
        ax.plot(lambdas, als_search[column], color=color, lw=2, marker="o", ms=8, mec=SURFACE, mew=1.5,
                solid_capstyle="round", solid_joinstyle="round", label=f"focal MF, {split}")
        ax.annotate(split, (lambdas[-1], als_search[column].iat[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color=INK_2)
    # The chosen lambda: a ring around its valid point.
    ax.plot(best_l2, als_search.loc[best_l2, f"valid {SELECT_BY}"], marker="o", ms=15, mfc="none", mec=INK, mew=1.2)
    ax.set_xscale("log")
    ax.set_xticks(lambdas, [f"{value:g}" for value in lambdas])
    ax.minorticks_off()
    ax.set(xlabel="λ (L2 penalty on the factors)", ylabel=metric_label,
           title=f"Focal MF, k = {FACTORS}: {metric_label} at the kept iteration (ring: chosen λ = {best_l2:g})")
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

## Test

The chosen models next to the global rate $\sigma(\mu)$, a constant probability (so it predicts every row as the same class and has ROC AUC 0.5).
Every model is scored per class (negative and positive) at `DECISION_THRESHOLD`, plus accuracy, macro F1, ROC AUC and average precision,
and the log losses of the probability. `test, recipe in train` keeps only the test rows whose recipe has a train row. On the other rows every model
gets its logit from $\mu$ and $b_u$ alone, so this subset shows what the factors add.


In [ ]:
bias_model, als_model = bias_runs[best_l2_bias][0], als_runs[best_l2][0]
test_users, test_recipes, test_values = encoded["test"]
in_train = test_recipes >= 0
eval_rows = {"valid": encoded["valid"], "test": encoded["test"],
             "test, recipe in train": (test_users[in_train], test_recipes[in_train], test_values[in_train])}
models = {
    f"global rate (σ(μ) = {expit(als_model['mu']):.2f})": lambda users, recipes: np.full(len(users), expit(als_model["mu"])),
    f"biases only (λ_b={best_l2_bias:g})": lambda users, recipes: predict(bias_model, users, recipes),
    f"focal MF k={FACTORS} (γ={FOCAL_GAMMA:g}, λ={best_l2:g}, λ_b={best_l2_bias:g})":
        lambda users, recipes: predict(als_model, users, recipes),
}
result_rows = {}
for model, score_rows in models.items():
    for split, (users, recipes, values) in eval_rows.items():
        score = score_rows(users, recipes)
        result_rows[(model, split)] = {**classification_metrics(values, score), **probability_metrics(values, score)}
results = pd.DataFrame(result_rows).T.rename_axis(["model", "split"])
display(results.round(4))
print(f"{(~in_train).mean():.1%} of test rows have a recipe with no train row.")


### How well each class is predicted

The chosen focal MF model on test, at `DECISION_THRESHOLD`: for each class, **precision** is the share of the rows predicted as that class
that belong to it, **recall** is the share of the class's rows predicted as it, and **F1** is their harmonic mean. The fourth bar, also
printed under each group, is the class's share of all labelled rows. The confusion matrix gives the counts, and the last table splits the
negatives into real low ratings (0–3★) and sampled unrated recipes, to show which kind the model catches.


In [ ]:
class_names = ["negative", "positive"]
true_label = test_values.astype(int)
test_score = predict(als_model, test_users, test_recipes)
predicted_label = (test_score >= DECISION_THRESHOLD).astype(int)
precision, recall, f1, support = precision_recall_fscore_support(true_label, predicted_label, labels=LABELS,
                                                                 zero_division=0)
per_class = pd.DataFrame({
    "precision": precision, "recall": recall, "f1": f1,
    "share of dataset": labelled["label"].value_counts(normalize=True).reindex(LABELS, fill_value=0).to_numpy(),
    "test rows": support, "predicted": np.bincount(predicted_label, minlength=len(LABELS)),
}, index=pd.Index(class_names, name="class"))
display(per_class.round(3))
display(pd.DataFrame(confusion_matrix(true_label, predicted_label, labels=LABELS),
                     index=[f"true {name}" for name in class_names],
                     columns=[f"predicted {name}" for name in class_names]))
test_source = splits["test"]["source"].to_numpy()
by_source = pd.DataFrame({
    kind: {"test rows": int(rows.sum()), "predicted positive": predicted_label[rows].mean(),
           "mean score": test_score[rows].mean()}
    for kind in pd.unique(test_source) for rows in [test_source == kind]}).T.rename_axis("test rows by source")
display(by_source.round(3))
test_metrics = classification_metrics(test_values, test_score)
print(f"Test accuracy {test_metrics['accuracy']:.3f}, macro F1 {test_metrics['macro f1']:.3f}, "
      f"ROC AUC {test_metrics['roc auc']:.3f}, average precision {test_metrics['average precision']:.3f} "
      f"(threshold {DECISION_THRESHOLD:g})")

positions = np.arange(len(LABELS))
bars = (("precision", BLUE, "precision"), ("recall", ORANGE, "recall"), ("f1", AQUA, "F1"),
        ("share of dataset", YELLOW, "share of the dataset"))
width, spacing = 0.15, 0.165  # Bar width, and the distance between bar centers (a thin gap between neighbours).
class_labels = [f"negative (0-{POSITIVE_MIN - 1}★ or unrated)", f"positive ({POSITIVE_MIN}-5★)"]
with plt.rc_context(VIZ_STYLE):
    fig, ax = plt.subplots(figsize=(8, 4.4))
    for k, (column, color, label) in enumerate(bars):
        ax.bar(positions + (k - (len(bars) - 1) / 2) * spacing, per_class[column], width=width, color=color, label=label)
    ax.set_xticks(positions, [f"{name}\n{share:.1%} of rows" for name, share in zip(class_labels, per_class["share of dataset"])])
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set(ylim=(0, 1.2), xlabel="class", ylabel="score or share",
           title=f"Focal MF on test: precision, recall and F1 by class (γ = {FOCAL_GAMMA:g}, WEIGHT_BALANCE = {WEIGHT_BALANCE:g})")
    ax.tick_params(axis="x", length=0)
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False, ncols=4, loc="upper left")
    fig.tight_layout()
    plt.show()


## Save the model

`model_artifacts/mf_als/` holds:

- `als_model.npz`: `mu`; `user_bias` and `user_factors`, one row per entry of `user_ids`; `item_bias` and `item_factors`, one row per entry of `recipe_ids`.
- `l2_search.csv` (both searches), `metrics.csv` (the test table), `per_class.csv` (precision, recall, and F1 per class), and
  `config.json` (data filters, labels, negative sampling, split, class weights, focal loss, threshold, and hyperparameters).

To score user $u$ on recipe $i$, add $\mu$, $b_u$, $c_i$, and $p_u^\top q_i$ (the logit), leaving out the terms of a user or recipe that
is not in the vocabulary, then apply the sigmoid. A probability of at least `DECISION_THRESHOLD` means positive (the user likes the recipe).


In [ ]:
ALS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(ALS_DIR / "als_model.npz", user_ids=user_ids, recipe_ids=recipe_ids, mu=als_model["mu"],
         user_bias=als_model["user_bias"], user_factors=als_model["user_factors"],
         item_bias=als_model["item_bias"], item_factors=als_model["item_factors"])
pd.concat({"biases only": bias_search.rename_axis("l2"), f"ALS k={FACTORS}": als_search.rename_axis("l2")},
          names=["model"]).to_csv(ALS_DIR / "l2_search.csv")
results.to_csv(ALS_DIR / "metrics.csv")
per_class.to_csv(ALS_DIR / "per_class.csv")
(ALS_DIR / "config.json").write_text(json.dumps({
    "task": "binary: positive (rating >= positive_min) vs negative; logistic biased matrix factorization",
    "loss": "class-weighted focal loss (1 - p_t)^gamma * BCE + L2, fitted by alternating Newton steps with a line search",
    "score": "sigmoid(mu + b_u + c_i + p_u . q_i)", "focal_gamma": FOCAL_GAMMA,
    "data": str(DATA_PATH.relative_to(NOTEBOOK_DIR)), "ratings_kept": "0-5",
    "positive_min": POSITIVE_MIN, "labels": {"0": f"negative: rating 0-{POSITIVE_MIN - 1} or a sampled unrated recipe",
                                             "1": f"positive: rating {POSITIVE_MIN}-5"},
    "negative_sampling": {"per_positive": NEGATIVES_PER_POSITIVE,
                          "pool": "recipes of the kept ratings, uniform, never rated by the user, no repeats per user",
                          "split": "the positive's split"},
    "min_user_ratings": MIN_USER_RATINGS,
    "split": "per user by date: latest rating -> test, the one before -> valid, the rest -> train",
    "users": len(user_ids), "recipes": len(recipe_ids), "train_rows": int(R.nnz), "train_positive": int(R.sum()),
    "weight_balance": WEIGHT_BALANCE, "class_weights": {str(c): float(class_weight[c]) for c in LABELS},
    "decision_threshold": DECISION_THRESHOLD, "min_curvature": MIN_CURVATURE, "line_search_steps": list(LINE_SEARCH_STEPS),
    "select_by": SELECT_BY, "factors": FACTORS, "l2": best_l2, "l2_bias": best_l2_bias,
    "l2_grid": L2_GRID, "l2_bias_grid": L2_BIAS_GRID, "iterations": int(als_model["iterations"]),
    "init_std": INIT_STD, "random_state": RANDOM_STATE,
}, indent=2))
print(f"Saved to {ALS_DIR}: {sorted(p.name for p in ALS_DIR.iterdir())}")
